# Thai Election OCR Pipeline — OpenRouter + Gemini 2.0 Flash
**Preprocessing** → **OCR (OpenRouter API)** → **Post-processing (fuzzy match)** → **CSV**

In [ ]:
# === Cell 1: Install ===
!pip install -q openai rapidfuzz tqdm Pillow pandas kaggle 2>/dev/null
print('Done.')

In [ ]:
# === Cell 2: Imports & Config ===
import os, re, json, time, base64, io
from pathlib import Path
from collections import defaultdict

import pandas as pd
from PIL import Image, ImageEnhance
from tqdm.notebook import tqdm
from openai import OpenAI
from rapidfuzz import fuzz
from google.colab import userdata

# === OpenRouter Config ===
OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
MODEL = 'google/gemini-2.0-flash-001'  # via OpenRouter

client = OpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=OPENROUTER_API_KEY,
)

# Paths
DATA_DIR = Path('/content/data')
IMAGE_DIR = DATA_DIR / 'images'
SAMPLE_LABELS_DIR = DATA_DIR / 'sample_labels'
SUBMISSION_TEMPLATE = DATA_DIR / 'submission_template.csv'
OUTPUT_DIR = Path('/content/output')
OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT_FILE = OUTPUT_DIR / 'ocr_results.json'

print(f'Model: {MODEL} via OpenRouter')

In [ ]:
# === Cell 3: Download Data ===
import os, json
os.makedirs('/root/.kaggle', exist_ok=True)
from google.colab import userdata
username = userdata.get('KAGGLE_USERNAME')
key = userdata.get('KAGGLE_KEY')
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': username, 'key': key}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

COMPETITION = 'super-ai-engineer-season-6-ocr-2569'
!kaggle competitions download -c {COMPETITION} -p /content/

import glob as glob_module
for zf in glob_module.glob('/content/*.zip'):
    !unzip -qo {zf} -d /content/data

# Verify
imgs = list(IMAGE_DIR.glob('*.png')) if IMAGE_DIR.exists() else []
print(f'Images: {len(imgs)}')
print(f'Template: {SUBMISSION_TEMPLATE.exists()}')
print(f'Sample labels: {SAMPLE_LABELS_DIR.exists()}')

In [ ]:
# === Cell 4: Group Documents & Build Expected Parties ===
template_df = pd.read_csv(SUBMISSION_TEMPLATE)
print(f'Template: {template_df.shape[0]} rows, {template_df.doc_id.nunique()} docs')

# Group images into documents
def group_documents(image_dir):
    docs = defaultdict(list)
    pattern = re.compile(r'^(.+?)(?:_page(\d+))?\.png$')
    for p in sorted(image_dir.glob('*.png')):
        m = pattern.match(p.name)
        if m:
            doc_id = m.group(1)
            page = int(m.group(2)) if m.group(2) else 1
            docs[doc_id].append((page, str(p)))
    for d in docs:
        docs[d].sort()
    return dict(docs)

documents = group_documents(IMAGE_DIR)
doc_types = {d: ('constituency' if d.startswith('constituency') else 'partylist') for d in documents}

# Expected parties per doc from template
doc_expected = defaultdict(list)
for _, row in template_df.iterrows():
    doc_expected[row['doc_id']].append({
        'id': row['id'],
        'party_name': row['party_name'],
        'row_num': row['row_num'],
    })

template_doc_rows = {d: len(rows) for d, rows in doc_expected.items()}

print(f'Documents: {len(documents)}')
print(f'Constituency: {sum(1 for t in doc_types.values() if t=="constituency")}')
print(f'Party list: {sum(1 for t in doc_types.values() if t=="partylist")}')

In [ ]:
# === Cell 5: Image Preprocessing ===

def preprocess_image(img_path, max_width=1024, quality=85):
    """Resize + enhance + encode to base64 JPEG.
    
    - Resize to max 1024px wide (saves tokens significantly)
    - Enhance contrast for scanned docs
    - Compress to JPEG (vs PNG = ~3-5x smaller)
    - Returns base64 string for API
    """
    img = Image.open(img_path)
    if img.mode != 'RGB':
        img = img.convert('RGB')
    
    # Resize
    w, h = img.size
    if w > max_width:
        ratio = max_width / w
        img = img.resize((max_width, int(h * ratio)), Image.LANCZOS)
    
    # Enhance for scanned documents
    img = ImageEnhance.Contrast(img).enhance(1.3)
    img = ImageEnhance.Sharpness(img).enhance(1.5)
    
    # Encode to JPEG base64
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=quality)
    b64 = base64.b64encode(buf.getvalue()).decode('utf-8')
    
    return b64, img.size

# Test
test_img = list(IMAGE_DIR.glob('*.png'))[0]
b64, size = preprocess_image(str(test_img))
orig = Image.open(test_img).size
print(f'Original: {orig[0]}x{orig[1]} | Processed: {size[0]}x{size[1]} | Base64: {len(b64)//1024}KB')

In [ ]:
# === Cell 6: Prompts ===

CONSTITUENCY_PROMPT = """คุณคือระบบ OCR สำหรับเอกสารผลเลือกตั้ง สส. แบบแบ่งเขต (สส.6/1)
ภาพเหล่านี้คือหน้าต่างๆ จากเอกสารเดียวกัน

ดึงข้อมูลจากตารางคะแนน ทุกแถว:
- ชื่อพรรคการเมือง (ภาษาไทย ตามที่ปรากฏในเอกสาร)
- คะแนน (ตัวเลข Arabic 0-9 ไม่มี comma)

กฎ:
- แปลงเลขไทย ๐-๙ เป็น 0-9
- ห้ามข้ามแถว แม้คะแนน 0
- รวมทุกหน้า ห้ามซ้ำ
- ไม่สนหัวตาราง/ลายเซ็น/แถวรวม

ตอบ JSON array:
[{\"party_name\": \"ชื่อพรรค\", \"votes\": \"12345\"}]

JSON เท่านั้น ห้ามมีข้อความอื่น"""

PARTYLIST_PROMPT = """คุณคือระบบ OCR สำหรับเอกสารผลเลือกตั้ง สส. แบบบัญชีรายชื่อ (สส.6/1)
ภาพเหล่านี้คือหน้าต่างๆ จากเอกสารเดียวกัน

ดึงข้อมูลจากตารางคะแนน ทุกแถว:
- ชื่อพรรคการเมือง (ภาษาไทย ตามที่ปรากฏในเอกสาร)
- คะแนน (ตัวเลข Arabic 0-9 ไม่มี comma)

กฎ:
- แปลงเลขไทย ๐-๙ เป็น 0-9
- ต้องมีประมาณ 57 พรรค
- รวมทุกหน้า ห้ามซ้ำ
- ห้ามข้ามแถว แม้คะแนน 0
- ไม่สนหัวตาราง/ลายเซ็น/แถวรวม

ตอบ JSON array:
[{\"party_name\": \"ชื่อพรรค\", \"votes\": \"12345\"}]

JSON เท่านั้น ห้ามมีข้อความอื่น"""

print('Prompts ready.')

In [ ]:
# === Cell 7: OCR Engine + Post-processing ===

THAI_TO_ARABIC = str.maketrans('\u0e50\u0e51\u0e52\u0e53\u0e54\u0e55\u0e56\u0e57\u0e58\u0e59', '0123456789')

def clean_votes(raw):
    """Clean vote string: Thai numerals -> Arabic, remove non-digits."""
    if not raw or raw == 'None':
        return '0'
    s = str(raw).translate(THAI_TO_ARABIC)
    s = re.sub(r'[^0-9]', '', s)
    return s if s else '0'


def extract_document(doc_id, page_paths, doc_type):
    """Send preprocessed images to OpenRouter, return JSON."""
    prompt = CONSTITUENCY_PROMPT if doc_type == 'constituency' else PARTYLIST_PROMPT
    
    # Build message with preprocessed images
    content = []
    for _, page_path in page_paths:
        b64, _ = preprocess_image(page_path)
        content.append({
            'type': 'image_url',
            'image_url': {'url': f'data:image/jpeg;base64,{b64}'}
        })
    content.append({'type': 'text', 'text': prompt})
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': content}],
        temperature=0.0,
        max_tokens=8192,
    )
    
    text = response.choices[0].message.content.strip()
    # Clean JSON
    if text.startswith('```'):
        text = re.sub(r'^```(?:json)?\n?', '', text)
        text = re.sub(r'\n?```$', '', text)
    
    rows = json.loads(text)
    return rows


def match_parties(ocr_rows, expected_rows):
    """Fuzzy match OCR party names to template party names.
    
    Returns dict: submission_id -> votes string
    """
    result = {}
    
    # Build OCR lookup: party_name -> votes
    ocr_lookup = {}
    for r in ocr_rows:
        name = r.get('party_name', '')
        votes = clean_votes(r.get('votes', '0'))
        if name:
            ocr_lookup[name] = votes
    
    used = set()
    
    for exp in expected_rows:
        exp_name = exp['party_name']
        sub_id = exp['id']
        
        # 1. Exact match
        if exp_name in ocr_lookup and exp_name not in used:
            result[sub_id] = ocr_lookup[exp_name]
            used.add(exp_name)
            continue
        
        # 2. Fuzzy match
        best_name = None
        best_score = 0
        for ocr_name in ocr_lookup:
            if ocr_name in used:
                continue
            score = max(
                fuzz.ratio(exp_name, ocr_name),
                fuzz.partial_ratio(exp_name, ocr_name),
                fuzz.token_sort_ratio(exp_name, ocr_name),
            )
            if score > best_score and score >= 60:
                best_score = score
                best_name = ocr_name
        
        if best_name:
            result[sub_id] = ocr_lookup[best_name]
            used.add(best_name)
        else:
            result[sub_id] = '0'
    
    return result


# === Checkpoint functions ===
def load_checkpoint():
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, encoding='utf-8') as f:
            return json.load(f)
    return {}

def save_checkpoint(results):
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

def save_submission_csv(results):
    """Generate submission CSV with fuzzy matching."""
    all_votes = {}
    for doc_id, expected_rows in doc_expected.items():
        ocr_rows = results.get(doc_id, [])
        if not ocr_rows:
            for exp in expected_rows:
                all_votes[exp['id']] = '0'
        else:
            matched = match_parties(ocr_rows, expected_rows)
            all_votes.update(matched)
    
    sub = template_df[['id']].copy()
    sub['votes'] = sub['id'].map(all_votes).fillna('0')
    csv_path = OUTPUT_DIR / 'submission.csv'
    sub.to_csv(csv_path, index=False)
    return csv_path

print('Engine ready: preprocess -> OCR -> fuzzy match -> CSV')

In [ ]:
# === Cell 8: Test on 1 document ===
test_doc = list(documents.keys())[0]
test_type = doc_types[test_doc]
expected = [e['party_name'] for e in doc_expected[test_doc]]
print(f'Testing: {test_doc} ({test_type}), {len(expected)} parties')

t0 = time.time()
ocr_rows = extract_document(test_doc, documents[test_doc], test_type)
elapsed = time.time() - t0
print(f'OCR: {len(ocr_rows)} rows in {elapsed:.1f}s')

# Show OCR output
for r in ocr_rows[:10]:
    print(f"  {r['party_name']:<30s} {r['votes']:>8s}")
if len(ocr_rows) > 10:
    print(f'  ... +{len(ocr_rows)-10} more')

# Fuzzy match test
matched = match_parties(ocr_rows, doc_expected[test_doc])
matched_count = sum(1 for v in matched.values() if v != '0')
print(f'\nMatched: {matched_count}/{len(expected)} parties')

# Compare with ground truth if available
label_file = SAMPLE_LABELS_DIR / f'{test_doc}.json'
if label_file.exists():
    with open(label_file, encoding='utf-8') as f:
        label = json.load(f)
    gt = {r.get('party', r.get('party_name', '')): str(r['votes']) for r in label['results']}
    print(f'\n--- Ground Truth Comparison ---')
    for exp in doc_expected[test_doc]:
        pred = matched.get(exp['id'], '0')
        actual = gt.get(exp['party_name'], '?')
        ok = '✓' if pred == actual else '✗'
        print(f"  {ok} {exp['party_name']:<25s} actual={actual:>8s} pred={pred:>8s}")

In [ ]:
# === Cell 9: Run All Documents ===
results = load_checkpoint()
print(f'Loaded {len(results)} existing results from checkpoint.')

doc_ids_sorted = sorted(documents.keys(), key=lambda d: (doc_types.get(d, 'z'), d))
todo = [d for d in doc_ids_sorted if d not in results]
print(f'To process: {len(todo)} (skipped {len(results)} done)')

SAVE_EVERY = 5
DELAY = 2  # OpenRouter has higher rate limits
errors = []
start_time = time.time()

for i, doc_id in enumerate(tqdm(todo, desc='OCR')):
    doc_type = doc_types.get(doc_id, 'constituency')
    page_paths = documents[doc_id]

    for attempt in range(3):
        try:
            rows = extract_document(doc_id, page_paths, doc_type)
            results[doc_id] = rows

            expected_count = template_doc_rows.get(doc_id, -1)
            if expected_count > 0 and len(rows) != expected_count:
                print(f'  WARN {doc_id}: got {len(rows)}, expected {expected_count}')
            break

        except json.JSONDecodeError as e:
            print(f'  JSON error {doc_id} (attempt {attempt+1}): {e}')
            time.sleep(5)
        except Exception as e:
            err_str = str(e)
            if '429' in err_str or 'rate' in err_str.lower():
                wait = 30 * (attempt + 1)
                print(f'  Rate limited, waiting {wait}s...')
                time.sleep(wait)
            else:
                print(f'  Error {doc_id} (attempt {attempt+1}): {e}')
                time.sleep(5)
    else:
        errors.append(doc_id)
        print(f'  FAILED: {doc_id}')

    # Periodic save
    done = len(results)
    if done % SAVE_EVERY == 0 or i == len(todo) - 1:
        save_checkpoint(results)
        save_submission_csv(results)
        elapsed = time.time() - start_time
        remaining = (elapsed / max(i+1, 1)) * (len(todo) - i - 1)
        print(f'  [{done}/{len(documents)}] {elapsed/60:.1f}min, ~{remaining/60:.0f}min left')

    time.sleep(DELAY)

# Final save
save_checkpoint(results)
save_submission_csv(results)
elapsed = time.time() - start_time
print(f'\nDone! {len(results)}/{len(documents)} docs in {elapsed/60:.1f} min')
if errors:
    print(f'Failed ({len(errors)}): {errors}')
print(f'Submission: {OUTPUT_DIR}/submission.csv')

In [ ]:
# === Cell 10: Validate Against Sample Labels ===
def levenshtein(s1, s2):
    if len(s1) < len(s2): return levenshtein(s2, s1)
    if len(s2) == 0: return len(s1)
    prev = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        curr = [i + 1]
        for j, c2 in enumerate(s2):
            curr.append(min(prev[j+1]+1, curr[j]+1, prev[j]+(c1!=c2)))
        prev = curr
    return prev[-1]

results = load_checkpoint()
total_dist = 0
total_pairs = 0

if SAMPLE_LABELS_DIR.exists():
    for jf in sorted(SAMPLE_LABELS_DIR.glob('*.json')):
        doc_id = jf.stem
        with open(jf, encoding='utf-8') as f:
            label = json.load(f)
        gt = {r.get('party', r.get('party_name', '')): str(r['votes']) for r in label['results']}
        ocr_rows = results.get(doc_id, [])
        matched = match_parties(ocr_rows, doc_expected.get(doc_id, []))
        
        print(f'\n=== {doc_id} ===')
        doc_dist = 0
        for exp in doc_expected.get(doc_id, []):
            pred = matched.get(exp['id'], '0')
            actual = gt.get(exp['party_name'], '0')
            dist = levenshtein(str(pred), str(actual))
            doc_dist += dist
            total_dist += dist
            total_pairs += 1
            ok = '✓' if dist == 0 else '✗'
            print(f"  {ok} {exp['party_name']:<35s} actual={actual:>8s} pred={pred:>8s} dist={dist}")
        print(f'  Doc mean dist: {doc_dist/max(len(doc_expected.get(doc_id,[])),1):.2f}')

if total_pairs > 0:
    print(f'\n=== Overall: mean Levenshtein = {total_dist/total_pairs:.4f} ({total_pairs} pairs) ===')

In [ ]:
# === Cell 11: Download Submission ===
from google.colab import files
files.download(str(OUTPUT_DIR / 'submission.csv'))